# M2 Round 3B — VAR Baseline with BIC Lag Selection

**Issue:** #29  
**Owner:** Mitchel  
**Reviewer:** Lerneir  
**Branch:** `artifact/m2-round3b-baseline-var-bic`

## Objective

This notebook implements Round 3 Path B of the baseline-model comparison.

The goal is to evaluate a Vector Autoregression (VAR) model using lag order selected by the Bayesian Information Criterion (BIC), and compare its forecast accuracy against the shared Random Walk (naïve) benchmark.

To preserve a controlled AIC-vs-BIC comparison with Issue #28, the dataset, feature set, transformations, forecast horizons, evaluation procedure, benchmark, and metrics should remain the same. The intended experimental difference is the lag-order selection criterion.

## Common Round 3 feature set

- `yield_spread_10y_2y`
- `overnight_rate`
- `usdcad`
- `us_treasury_10y`
- `fed_funds_rate`
- `cpi_yoy`

## Round 2 findings carried into this notebook

The merged Round 2 EDA found that the candidate series are non-stationary in levels and stationary after first differencing. Therefore, the VAR baseline is estimated using stationarity-corrected inputs rather than raw levels.

The Round 2 redundancy check also supports keeping both U.S. drivers:

- `corr(Δus_treasury_10y, Δfed_funds_rate) = 0.044`

This indicates that the U.S. long-term Treasury yield and the Fed Funds Rate are not redundant at daily frequency.

## Forecast evaluation

Forecast horizons:

- 1 observation ahead
- 5 observations ahead
- 20 observations ahead

Metrics:

- RMSE
- MAE

Benchmark:

- Random Walk / naïve forecast

## Path B distinction

This notebook selects the VAR lag order using **BIC**.

Issue #28 uses **AIC**.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.api import VAR

# Resolve project root from notebooks/03_models/
PROJECT_ROOT = Path.cwd().resolve().parents[1]
PROCESSED = PROJECT_ROOT / "data" / "processed"

# Shared Round 3 feature set
FEATURES = [
    "yield_spread_10y_2y",
    "overnight_rate",
    "usdcad",
    "us_treasury_10y",
    "fed_funds_rate",
    "cpi_yoy",
]

TARGET = "yield_spread_10y_2y"

HORIZONS = [1, 5, 20]

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED)

Project root: C:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5
Processed data: C:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\data\processed


In [2]:
# Load processed datasets
boc = pd.read_csv(
    PROCESSED / "bank_of_canada_data.csv",
    parse_dates=["date"]
)

fred = pd.read_csv(
    PROCESSED / "fred_rates.csv",
    parse_dates=["date"]
)

cpi = pd.read_csv(
    PROCESSED / "statcan_cpi.csv",
    parse_dates=["reference_month", "release_date"]
)

print("BoC:", boc.shape)
print("FRED:", fred.shape)
print("CPI:", cpi.shape)

BoC: (4552, 12)
FRED: (4563, 3)
CPI: (210, 3)


In [3]:
# Merge BoC + FRED on date
daily = (
    boc.merge(fred, on="date", how="outer")
       .sort_values("date")
       .reset_index(drop=True)
)

# Compute CPI YoY at monthly frequency BEFORE expanding to daily
cpi_monthly = cpi.sort_values("reference_month").copy()
cpi_monthly["cpi_yoy"] = (
    cpi_monthly["cpi_all_items"].pct_change(12) * 100
)

# Align CPI by release date to avoid look-ahead bias
cpi_daily = (
    cpi_monthly[
        ["release_date", "cpi_all_items", "cpi_yoy"]
    ]
    .rename(columns={"release_date": "date"})
    .sort_values("date")
)

daily = (
    daily.merge(cpi_daily, on="date", how="left")
         .sort_values("date")
         .reset_index(drop=True)
)

# Once CPI is publicly available, carry the latest released value forward
daily["cpi_all_items"] = daily["cpi_all_items"].ffill()
daily["cpi_yoy"] = daily["cpi_yoy"].ffill()

print("Daily merged shape:", daily.shape)
print("Date range:", daily["date"].min(), "->", daily["date"].max())

daily[
    [
        "date",
        "yield_spread_10y_2y",
        "overnight_rate",
        "usdcad",
        "us_treasury_10y",
        "fed_funds_rate",
        "cpi_yoy",
    ]
].tail()

Daily merged shape: (4563, 16)
Date range: 2009-01-02 00:00:00 -> 2026-06-30 00:00:00


,date,yield_spread_10y_2y,overnight_rate,usdcad,us_treasury_10y,fed_funds_rate,cpi_yoy
4558,2026-06-24,0.63,2.25,1.4234,4.41,3.63,3.225806
4559,2026-06-25,0.64,2.25,1.4204,4.40,3.63,3.225806
4560,2026-06-26,0.64,2.25,1.4186,4.38,3.63,3.225806
4561,2026-06-29,0.64,2.25,1.4206,4.38,3.63,3.225806
4562,2026-06-30,0.64,2.25,1.4210,4.44,3.63,3.225806


In [4]:
# Keep only the agreed Round 3 feature set
model_levels = (
    daily[["date"] + FEATURES]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

# First-difference each series once, following the Round 2 ADF findings
model_diff = model_levels.copy()

for col in FEATURES:
    model_diff[col] = model_diff[col].diff()

# Remove rows that are unavailable after alignment/differencing
model_diff = (
    model_diff
    .dropna(subset=FEATURES)
    .reset_index(drop=True)
)

print("Modeling sample shape:", model_diff.shape)
print("Modeling date range:", model_diff["date"].min(), "->", model_diff["date"].max())

model_diff.head()

Modeling sample shape: (3774, 7)
Modeling date range: 2010-02-19 00:00:00 -> 2026-06-30 00:00:00


,date,yield_spread_10y_2y,overnight_rate,usdcad,us_treasury_10y,fed_funds_rate,cpi_yoy
0,2010-02-19,-0.01,0.0,-0.0032,-0.01,0.01,0.0
1,2010-02-22,0.01,0.0,0.0008,0.02,-0.01,0.0
2,2010-02-23,-0.02,0.0,0.0089,-0.11,0.00,0.0
3,2010-02-24,0.01,0.0,0.0033,0.01,-0.01,0.0
4,2010-02-25,0.01,0.0,0.0125,-0.06,0.01,0.0


In [5]:
# Separate dates from stationary model inputs
model_dates = model_diff["date"].copy()

var_data = (
    model_diff[FEATURES]
    .copy()
)

print("VAR input shape:", var_data.shape)
print("Columns:", list(var_data.columns))

VAR input shape: (3774, 6)
Columns: ['yield_spread_10y_2y', 'overnight_rate', 'usdcad', 'us_treasury_10y', 'fed_funds_rate', 'cpi_yoy']


In [7]:
# Select VAR lag order using BIC
MAX_LAGS = 15

lag_selection = VAR(var_data).select_order(maxlags=MAX_LAGS)

print(lag_selection.summary())
print("\nSelected lag by BIC:", lag_selection.bic)

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -41.38     -41.37*   1.072e-18     -41.37*
1       -41.38      -41.31   1.073e-18      -41.35
2       -41.38      -41.25   1.073e-18      -41.33
3       -41.38      -41.19   1.071e-18      -41.31
4       -41.37      -41.12   1.079e-18      -41.28
5       -41.39      -41.08   1.060e-18      -41.28
6       -41.38      -41.01   1.069e-18      -41.25
7       -41.37      -40.95   1.075e-18      -41.22
8       -41.38      -40.89   1.071e-18      -41.20
9       -41.38      -40.83   1.073e-18      -41.18
10     -41.51*      -40.91  9.361e-19*      -41.30
11      -41.51      -40.84   9.410e-19      -41.27
12      -41.50      -40.77   9.514e-19      -41.24
13      -41.50      -40.71   9.525e-19      -41.22
14      -41.49      -40.64   9.591e-19      -41.19
15      -41.51      -40.60   9.404e-19      -41.19
-------------------------------

In [8]:
# Shared evaluation configuration
MIN_TRAIN = 500
STEP = 5

BIC_LAG = int(lag_selection.bic)

print("BIC lag:", BIC_LAG)
print("Minimum training observations:", MIN_TRAIN)
print("Evaluation step:", STEP)
print("Horizons:", HORIZONS)

BIC lag: 0
Minimum training observations: 500
Evaluation step: 5
Horizons: [1, 5, 20]


In [10]:
# Align target levels exactly to the final modeling dates
aligned_levels = (
    model_levels[
        model_levels["date"].isin(model_diff["date"])
    ][["date", TARGET]]
    .sort_values("date")
    .reset_index(drop=True)
)

# Safety check: dates must match row-by-row
assert len(aligned_levels) == len(model_diff)
assert aligned_levels["date"].equals(model_diff["date"])

print("Aligned observations:", len(aligned_levels))
print(
    "Aligned date range:",
    aligned_levels["date"].min(),
    "->",
    aligned_levels["date"].max()
)

Aligned observations: 3774
Aligned date range: 2010-02-19 00:00:00 -> 2026-06-30 00:00:00


In [11]:
results = []

target_idx = FEATURES.index(TARGET)

for origin in range(MIN_TRAIN, len(model_diff) - max(HORIZONS), STEP):
    train_diff = model_diff.iloc[:origin][FEATURES].copy()

    # Level corresponding exactly to the forecast origin
    last_level = aligned_levels.loc[origin - 1, TARGET]

    if BIC_LAG == 0:
        # VAR(0): constant expected change estimated from training data
        target_mean_change = train_diff[TARGET].mean()

    else:
        var_model = VAR(train_diff)
        var_fit = var_model.fit(BIC_LAG)

    for h in HORIZONS:
        # h observations ahead from the origin
        actual_level = aligned_levels.loc[origin + h - 1, TARGET]

        # Random Walk
        naive_forecast = last_level

        # VAR-BIC
        if BIC_LAG == 0:
            var_forecast = last_level + h * target_mean_change
        else:
            forecast_diff = var_fit.forecast(
                train_diff.values[-BIC_LAG:],
                steps=h
            )

            cumulative_target_change = forecast_diff[:, target_idx].sum()
            var_forecast = last_level + cumulative_target_change

        results.append({
            "origin_date": aligned_levels.loc[origin - 1, "date"],
            "horizon": h,
            "actual": actual_level,
            "var_bic": var_forecast,
            "naive": naive_forecast,
        })

results_df = pd.DataFrame(results)

print("Forecast rows:", len(results_df))
print(results_df.groupby("horizon").size())

results_df.head()

Forecast rows: 1953
horizon
1     651
5     651
20    651
dtype: int64


,origin_date,horizon,actual,var_bic,naive
0,2012-04-12,1,0.79,0.817720,0.82
1,2012-04-12,5,0.71,0.808600,0.82
2,2012-04-12,20,0.75,0.774400,0.82
3,2012-04-19,1,0.71,0.707525,0.71
4,2012-04-19,5,0.68,0.697624,0.71


In [12]:
metrics = []

for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h]

    var_rmse = np.sqrt(
        mean_squared_error(subset["actual"], subset["var_bic"])
    )
    var_mae = mean_absolute_error(
        subset["actual"], subset["var_bic"]
    )

    naive_rmse = np.sqrt(
        mean_squared_error(subset["actual"], subset["naive"])
    )
    naive_mae = mean_absolute_error(
        subset["actual"], subset["naive"]
    )

    metrics.append({
        "horizon": h,
        "var_bic_rmse": var_rmse,
        "naive_rmse": naive_rmse,
        "var_bic_mae": var_mae,
        "naive_mae": naive_mae,
        "rmse_improvement_pct": (
            (naive_rmse - var_rmse) / naive_rmse * 100
        ),
        "mae_improvement_pct": (
            (naive_mae - var_mae) / naive_mae * 100
        ),
    })

metrics_df = pd.DataFrame(metrics)

metrics_df.round(6)

,horizon,var_bic_rmse,naive_rmse,var_bic_mae,naive_mae,rmse_improvement_pct,mae_improvement_pct
0,1,0.030302,0.030268,0.021908,0.021751,-0.113554,-0.720553
1,5,0.065452,0.065181,0.048325,0.048111,-0.415446,-0.445418
2,20,0.137259,0.134971,0.103700,0.101613,-1.695819,-2.053665


In [13]:
# Save Round 3B results
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

metrics_output = OUTPUT_DIR / "r3_pathb_var_bic_vs_naive.csv"
forecasts_output = OUTPUT_DIR / "r3_pathb_var_bic_forecasts.csv"

metrics_df.to_csv(metrics_output, index=False)
results_df.to_csv(forecasts_output, index=False)

print("Saved metrics:", metrics_output)
print("Saved forecasts:", forecasts_output)

Saved metrics: C:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\outputs\r3_pathb_var_bic_vs_naive.csv
Saved forecasts: C:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\outputs\r3_pathb_var_bic_forecasts.csv


## Round 3B Conclusion — VAR with BIC Lag Selection

Using the common Round 3 feature set, all six variables were modeled in first differences based on the Round 2 stationarity findings. The final modeling sample contained 3,774 aligned daily observations.

For the BIC-based VAR specification, lag-order selection over a maximum of 15 lags selected:

- **BIC lag order: 0**

This indicates that, under the BIC penalty, adding autoregressive lags did not provide enough additional explanatory value to justify the increase in model complexity.

The BIC-selected specification was evaluated using an expanding-window approach with forecasts generated every 5 observations after an initial training sample of 500 observations. Performance was assessed at **1-day, 5-day, and 20-day horizons** using RMSE and MAE, with a Random Walk forecast used as the shared naïve benchmark.

### Forecast performance

| Horizon | VAR-BIC RMSE | Random Walk RMSE | VAR-BIC MAE | Random Walk MAE |
|---|---:|---:|---:|---:|
| 1 day | 0.030302 | 0.030268 | 0.021908 | 0.021751 |
| 5 days | 0.065452 | 0.065181 | 0.048325 | 0.048111 |
| 20 days | 0.137259 | 0.134971 | 0.103700 | 0.101613 |

The BIC specification did **not outperform the Random Walk benchmark at any forecast horizon**. Relative to the naïve benchmark, the VAR-BIC model produced slightly higher errors:

- **1-day horizon:** RMSE +0.11%, MAE +0.72%
- **5-day horizon:** RMSE +0.42%, MAE +0.45%
- **20-day horizon:** RMSE +1.70%, MAE +2.05%

Overall, BIC strongly favored model parsimony and selected a zero-lag specification. The resulting forecasts were very close to the Random Walk benchmark, but the Random Walk remained marginally more accurate across all three horizons. These results provide the Path B baseline for the team comparison between **AIC- and BIC-based VAR lag selection**.